### Installation

In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [3]:
!pip install modelscope

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 136.0 MB/s eta 0:00:00


In [4]:
import os; os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

In [ ]:
from unsloth import FastModel
import torch


model, tokenizer = FastModel.from_pretrained(
    model_name = "CohereLabs/tiny-aya-base",
    max_seq_length = 1024, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False,
    token = "", # HF Token for gated models,

)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


2026-03-21 07:19:59,654 - modelscope - INFO - Got 13 files, start to download ...


Processing 13 items:   0%|          | 0.00/13.0 [00:00<?, ?it/s]

We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 16,           # Rank — 16 is the sweet spot for 3B models on T4
    lora_alpha = 32,  # Scaling factor — 2x rank is standard
    lora_dropout = 0.00,# A regularization technique that helps prevent overfitting by randomly setting a fraction of the LoRA activations to zero during each training step. Recent research suggests that for the short training runs common in fine-tuning, lora_dropout may be an unreliable regularizer.
    bias = "none",
    random_state = 3407,
    gradient_checkpointing = "unsloth", #it reduces memory usage by an extra 30%,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj",]

)


We now use the `Gemma-3` format for conversation style finetunes. Gemma-3 renders multi turn conversations like below:
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>


In [ ]:
"""
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)
"""

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset(
    "legesher/language-decoded-data",
    "condition-1-en",
    split="train"
)

eval_dataset = load_dataset(
    "legesher/language-decoded-data",
    "condition-1-en",
    split="validation"
)

In [ ]:
train_dataset[100]
eval_dataset[100]

We now have to apply the chat template for `Gemma-3` onto the conversations, and save it to `text`. We remove the `<bos>` token using removeprefix(`'<bos>'`) since we're finetuning. The Processor will add this token before training and the model expects only one.


In [ ]:
def formatting_prompts_func(examples):
   code = examples["code"]
   return { "code" : code }

train_dataset = train_dataset.map(formatting_prompts_func, batched = True)

In [ ]:
train_dataset[100]["code"]
eval_dataset[100]["code"]

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "code",
        max_grad_norm = 1.0, # Gradient clipping
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2, # Use GA to mimic batch size!
        warmup_ratio = 0.05, # 5% warmup — ~308 steps for 49.5K files
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 3,
        learning_rate = 5e-5,  # Conservative for QLoRA continued pre-training
                                       # Lower than LoRA (2e-4) due to quantization noise
                              # Higher than Unsloth's 5e-6 since we have ~50K files

        packing= True,
        logging_steps = 1,
        optim = "paged_adamw_8bit",            # 8-bit Adam — saves ~1GB VRAM
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",# Cosine decay — smooth annealing
        seed = 42,
        report_to = "none", # Use TrackIO/WandB etc
        fp16 = True,
        bf16 = False,
        eval_strategy = "steps",
        eval_steps = 500,                    # Eval every 500 steps during training
        run_name = "condition-1-en"
    ),
)

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [ ]:
from unsloth.chat_templates import train_on_responses_only
"""
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)
"""

Let's verify masking the instruction part is done! Let's print the 100th row again.  Notice how the sample only has a single `<bos>` as expected!

In [ ]:
#tokenizer.decode(trainer.train_dataset[100]["input_ids"])

Now let's print the masked out example - you should see only the answer is present:

In [ ]:
#tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")